## Preprocessing

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "workflows"))
from c21_surrogate_training import run_preprocessing

# ── Parameters ───────────────────────────────────────────────────────────────
EDGE_CSV        = "v6_edge_C14_S19999_D20260516"
NODE_CSV        = "v6_node_C12_S19999_D20260516"
BIDIRECTIONAL   = False    # True=240 edges (undirected), False=120 edges (best)
BATCH_SIZE      = 32
DATA_INSPECTION = False    # print per-sample statistics

# Model architecture — smaller than v6 to prevent overfitting with WeightedBCE
# hidden_dim=128 / 4 layers had ~4M params and overfitted at epoch 21.
# hidden_dim=64  / 3 layers has ~800K params → can train 100+ epochs without overfitting.
HIDDEN_DIM      = 64       # was 128; reduces EdgeMLP params 4×
NUM_LAYERS      = 3        # was 4
DROPOUT_P       = 0.3      # was 0.1; stronger regularisation

# ── Run ───────────────────────────────────────────────────────────────────────
pre = run_preprocessing(
    EDGE_CSV, NODE_CSV,
    bidirectional   = BIDIRECTIONAL,
    batch_size      = BATCH_SIZE,
    data_inspection = DATA_INSPECTION,
    hidden_dim      = HIDDEN_DIM,
    num_layers      = NUM_LAYERS,
    dropout_p       = DROPOUT_P,
)

c:\Users\Jasper\Documents\PyEnvs\thesis_home_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config System loaded successfully, Code running locally from thesis_generative_timber and Data is connected to OneDrive 2.2 - 2.4.

Sample ID column: 'sample_id'
Topology: 120 edges in JSON | 120 rows/sample in CSV | 39 nodes/sample.
Using directed graph: 120 edges.
Final edge count: 120 (unidirectional).
Found 20000 matching samples.
Split: Train=16000 | Val=2000 | Test=2000
Norm stats saved: C:\Users\Jasper\Documents\PyRepo\thesis_generative_timber\02_data_io\norm_stats.pt
Train positive rate: 0.1958 → pos_weight=4.1080


## Training

In [ ]:
from c21_surrogate_training import run_training

# ── Parameters ───────────────────────────────────────────────────────────────
EPOCHS            = 200
LR                = 1e-4       # was 3e-4; lower LR needed for smaller model
PATIENCE          = 60         # was 40; allow more exploration before stopping
LR_FACTOR         = 0.5
LR_PATIENCE       = 15         # was 10
LR_MIN            = 1e-6
GRAD_CLIP         = 1.0
WEIGHT_DECAY      = 1e-3       # was 1e-4; stronger L2 prevents overfitting
DEFAULT_THRESHOLD = 0.35
MIN_PRECISION     = 0.40
# pos_weight is auto-computed from training data: (1 - pos_rate) / pos_rate ≈ 4.1

# ── Run ───────────────────────────────────────────────────────────────────────
train_out = run_training(
    pre,
    epochs            = EPOCHS,
    lr                = LR,
    patience          = PATIENCE,
    lr_factor         = LR_FACTOR,
    lr_patience       = LR_PATIENCE,
    lr_min            = LR_MIN,
    grad_clip         = GRAD_CLIP,
    weight_decay      = WEIGHT_DECAY,
    default_threshold = DEFAULT_THRESHOLD,
    min_precision     = MIN_PRECISION,
)

## Evaluation

In [ ]:
from c21_surrogate_training import run_evaluation

# ── Run ───────────────────────────────────────────────────────────────────────
eval_out = run_evaluation(train_out, pre)


## Export

In [ ]:
from c21_surrogate_training import run_export

# ── Run ───────────────────────────────────────────────────────────────────────
export_out = run_export(pre, train_out, eval_out)
print("Saved to:", export_out["models_dir"])
print("Files:")
for f in export_out["all_files"]:
    print(" ", f)
